# RAG with and without reranking

See how reranking changes the selected passages and whether that changes the generated answer.

**Recorded reference run:** 10 November 2026. Saved displays belong to that execution. A new Live run computes its own results; use its current tables rather than treating reference findings as predictions.

## One question, two RAG paths

**Recorded reference run:** the question was **“What is the cause of COVID-19?”**. Each path selects five passages from the same 30 candidates and uses the same prompt template and generator settings.

| Component | Model or setup |
| --- | --- |
| Embedding | `qwen/qwen3-embedding-4b` · 2,560 dimensions |
| Reranker | `qwen/qwen3-reranker-8b` |
| Generator | `qwen/qwen3-30b-a3b-instruct-2507` |
| Dataset | MS MARCO v2.1 validation (`microsoft/ms_marco`) |
| Pipeline | Haystack: query embedding → retrieve 30 passages → select five, or rerank then select five → generate answer |


Live uses the question and corpus declared by the selected input profile, printed in the code cells. The small profile uses a covered dataset question; its results are a separate exercise.

## Setup and live execution

Use the project's pinned environment and dependency lock. Live execution is the default: `MODE="live"` runs query embedding, retrieval, reranking, generation and scoring. Set `OPENROUTER_API_KEY` before starting the kernel. See [setup.md](../../setup.md) for environment, kernel and launch instructions.

The index covers the frozen **152,576-passage validation subset**, not the full validation corpus. Document embedding remains stopped at this snapshot. Exact Python and package versions are specified in the repository lock.

In [1]:
STUDY = 'demo'
from pathlib import Path
import os, sys
from IPython.display import display, Markdown, HTML
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "configs/tutorial_inputs.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.live_tutorial import TutorialRun
from shared_scoring.provider import embedding_payload, rerank_payload
from shared_scoring.tutorial_views import TutorialViews
from shared_scoring.notebook_display import display_table, details_html
from shared_scoring.io import read_json
import inspect, json
import pandas as pd

MODE = os.environ.get("NOTEBOOK_MODE", "live").lower()  # live | replay | plan
RUN_ID = os.environ.get("TUTORIAL_RUN_ID") or None
RESUME = os.environ.get("TUTORIAL_RESUME") == "1"  # explicit same-run resume only

tutorial = TutorialRun(PROJECT, STUDY, mode=MODE, run_id=RUN_ID, resume=RESUME)
tutorial.preflight(create_client=False)  # assets, credentials and offline tests; no model requests
client = None
if MODE == "live":
    from openai import OpenAI
    import httpx
    # Standard OpenAI-style client; the transport records this run and enforces caps.
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=tutorial.retrieval["api_base"],
        max_retries=0,
        timeout=tutorial.retrieval["timeout_seconds"],
        http_client=httpx.Client(transport=tutorial.transport()),
    )
    tutorial.bind_client(client)
# Selecting Live authorizes this notebook's declared workflow; no second paid-call switch.

view = TutorialViews(tutorial)
view.show("configuration")
from haystack import Pipeline, component
demo_question = tutorial.questions()[0] if MODE != "plan" else tutorial.declarations["demo_question"]
question = demo_question["question"]
demo_query_id = demo_question["query_id"]


Setting,Value
Embedding model,qwen/qwen3-embedding-4b
Embedding dimensions,2560
Reranker model,qwen/qwen3-reranker-8b
Generator model,qwen/qwen3-30b-a3b-instruct-2507
Candidate / context budget,30 / 5 passages
Eligible questions in this experiment,1
Generation limit,400 tokens


## Dataset

The corpus uses MS MARCO v2.1 validation. The following row illustrates the dataset structure; the full dataset is not bundled.


In [2]:
view.show('corpus')


{'contract': {'dataset': {'contract': {'dataset': {'name': 'microsoft/ms_marco',
     'configuration': 'v2.1',
     'split': 'validation',
     'revision': 'a47ee7aae8d7d466ba15f9f0bfac3b3681087b3a'},
    'passage_char_budget': 1800,
    'scope': {'mode': 'checkpointed_passages',
     'source_directory': '.cache/qwen-runs/full-validation/9d5b5fc55d9fea605a3e',
     'eligible_question_ids': 'reports/embedding-coverage-20261015T204959667725Z/fully-covered-question-ids.txt',
     'coverage_file': 'reports/embedding-coverage-20261015T204959667725Z/question-coverage.jsonl',
     'eligible_ids_sha256': 'cfd29784dae3dfc6c14c1be0ef7600ef4921c0771d4b18cac7450473e102144e',
     'coverage_sha256': '4cc0c24547dfea2ea7fbffd45215f33828b9a4d842064f0d192a7993001dc619',
     'eligible_question_count': 15585,
     'passage_count': 152576,
     'passage_embedding_calls_allowed': False,
     'baseline_questions': 31,
     'baseline_seed': 42,
     'baseline_diagnostic_questions': 1}},
   'validation_quest

Inspect a question, its passages, source URLs, support annotations and reference answers. This Rachel Carson example explains the schema; the pipeline demonstration uses COVID-19.

In `is_selected`, **1** marks a passage annotated as supporting the reference answer. These dataset labels let us evaluate retrieval coverage.


In [3]:
if MODE != "plan":
    query_file = tutorial.upstream("prepared_retrieval") / "queries.parquet"
    if query_file.exists():
        schema_examples = pd.read_parquet(query_file)
        example = schema_examples[schema_examples.question.str.contains("Rachel Carson", case=False, na=False)]
        display(example.head(1) if not example.empty else schema_examples.head(1))


,query_row,query_id,question,known_supporting_passage_ids,has_known_support_labels,reference_answers,original_answers,well_formed_answers
1,1,1102431,why did rachel carson write an obligation to e...,"[p_a7d7c1627719a2739347, p_df702fcc14ec0419b842]",True,[Rachel Carson writes The Obligation to Endure...,[Rachel Carson writes The Obligation to Endure...,[]


Each candidate retains its passage text, source metadata, embedding rank and similarity score. The corpus contains deduplicated passages with stable IDs and a 1,800-character passage limit. The retrieval section below displays the thirty candidates after retrieval has completed.


In [4]:
@component
class RetrieveQuestion:
    @component.output_types(question=str)
    def run(self, question: str):
        if question != tutorial.questions()[0]["question"]:
            raise ValueError("Update demo_question in the selected input profile before changing the pipeline question.")
        tutorial.retrieve(embed_api)
        return {"question": question}


## Query embedding and retrieval

Document embeddings and the FAISS index already exist for the frozen 152,576-passage corpus. The workflow embeds the question, retrieves thirty candidates, and produces separate top-five contexts and generator responses.

In [5]:
def embed_api(questions):
    # Actual OpenAI-style embedding call. Passage vectors remain prepared inputs.
    return client.embeddings.create(**embedding_payload(
        tutorial.retrieval["embedding_model"], questions, tutorial.retrieval["dimensions"],
        instruction=tutorial.retrieval["query_instruction"]))

display({"question": question, "candidates": 30, "passages_per_answer": 5})


{'question': 'What is the cause of COVID-19?',
 'candidates': 30,
 'passages_per_answer': 5}

The following cell shows the Haystack retrieval pipeline over the frozen **152,576 cached passages**. Inspect the thirty candidates after retrieval. Execution requires the corresponding corpus/index and matching configuration.

In [6]:
retrieval_pipeline = Pipeline()
retrieval_pipeline.add_component("retrieve", RetrieveQuestion())
if MODE != "plan":
    retrieval_pipeline.run({"retrieve": {"question": question}})
    view.show("contexts", query_id=demo_query_id)


retrieve: saved for CURRENT LIVE RUN 20261111T025839Z-db04c3e0ca6e


passage_id,rank,is_known_support,text,similarity_score
p_3728f219848bf793cb32,1,False,"The truth that we must stare down is that cancer and infections cannot be separated from each other. Cancer—always believed to be caused by genetic cell mutations—can in reality be caused by infections from viruses, bacteria, and fungi.",0.4901
passage_id,rank,is_known_support,text,similarity_score
p_39df7d1187fcc5489493,2,False,"Bacteria are the most common cause of infective endocarditis, especially streptococci, staphylococci, and enterococci. Although fungi and viruses can cause infective endocarditis, they are not the most frequent cause. Infective endocarditis is caused by infection with microorganisms, not an autoimmune response. A client develops inflammation of the endocardium caused by Staphylococcus aureus.",0.4684
p_de76223c44b3e69cd180,3,False,"California is a state, so it doesn't think! :-)) Just kidding. Cancer is caused by a genetic mutation. Genetic mutations are caused by UV light (sun, sun lamps, or tanning beds), some viruses, chemicals, air pollution, and other forms of radiation, like too much X-ray. Mutations also occur randomly in the DNA.",0.4676
p_8729b7573df5124d2792,4,False,"Hand, foot, and mouth disease is generally caused by coxsackievirus A-16, which is a member of the enterovirus family. There are other types of enteroviruses that can cause the symptoms as well, but these are less common.",0.4618
p_2c34293f7e92f17e5467,5,False,"You can also have it and not know it. Doctors call this “walking pneumonia.” If your pneumonia is caused by a bacteria or virus, you can spread it to someone else. Lifestyle habits, like smoking cigarettes and drinking too much alcohol, can also raise your chances of getting pneumonia. Causes. Bacteria, viruses, or fungi can cause pneumonia. Top causes include: Flu viruses; Cold viruses; RSV virus (the top cause of pneumonia in babies age 1 or younger; Bacteria called Streptococcus pneumoniae and Mycoplasma pneumoniae",0.4598
p_a727a7d1be3cf3c5a664,6,False,"HIV-1, which causes almost all the cases of AIDS worldwide; HIV-2, which causes an AIDS-like illness. HIV-2 infection is uncommon in North America. What causes HIV? HIV infection is caused by the human immunodeficiency virus. You can get HIV from contact with infected blood, semen, or vaginal fluids.",0.4564
p_0ff8dfc59276423c2b92,7,False,"Streptococcal infections are caused by any one of several species of Streptococcus. Different groups of these bacteria are spread in different ways—for example, through coughing or sneezing, through contact with infected wounds or sores, or during vaginal delivery (from mother to child).",0.4546
p_36c253c659a76c80373a,8,False,"In most cases, acute pancreatitis is caused by gallstones or heavy alcohol use. Other causes include medications, autoimmune disease, infections, trauma, metabolic disorders, and surgery. In up to 15% of people with acute pancreatitis, the cause is unknown. In about 70% of people, chronic pancreatitis is caused by long-time alcohol use. Other causes include gallstones, hereditary disorders of the pancreas, cystic fibrosis, high triglycerides, and certain medicines.",0.4543
p_44a4729531569405d184,9,False,"The cause of cholera is infection by the V. cholera bacteria. These bacteria were discovered in 1883. The German bacteriologist, Robert Koch (1843-1910), studied the disease during an epidemic in Egypt.",0.4526


## Embedding RAG

Select embedding ranks 1–5 from the same thirty candidates. The Haystack pipeline connects this selection to the prompt builder and generator.

Inspect the shared prompt template, followed by this path’s five-passage context and generated answer. Both RAG paths use the same template and generator settings.

In [7]:
def generate_api(messages, settings, routing=None):
    # The actual SDK call; the tutorial invokes it for each declared question/arm.
    return client.chat.completions.create(
        model=settings["model"], messages=messages,
        temperature=settings["temperature"], max_tokens=settings["max_tokens"],
        **settings["extras"], **({"extra_body": {"provider": routing}} if routing else {}))
@component
class GenerateAnswer:
    def __init__(self, arm):
        self.arm = arm
    @component.output_types(answer=str)
    def run(self, question: str):
        if question != tutorial.questions()[0]["question"]:
            raise ValueError("Update demo_question in the selected input profile before changing the pipeline question.")
        tutorial.generate(generate_api, arm=self.arm)
        answers = view.answers()
        return {"answer": answers.loc[answers.arm.eq(self.arm), "answer"].iloc[0]}

embedding_pipeline = Pipeline()
embedding_pipeline.add_component("generate", GenerateAnswer("baseline"))


🚅 Components
  - generate: GenerateAnswer
🛤️ Connections

In [8]:
if MODE != "plan":
    display(embedding_pipeline.run({"generate": {"question": question}}))
    view.show("walkthrough", query_id=demo_query_id)


generate-baseline: saved for CURRENT LIVE RUN 20261111T025839Z-db04c3e0ca6e


{'generate': {'answer': 'The cause of COVID-19 is a virus, specifically the SARS-CoV-2 virus. This is supported by the general understanding that viruses can cause infectious diseases, as evidenced by the passage stating that "bacteria, viruses, or fungi can cause pneumonia," and that "flu viruses" and "cold viruses" are among the causes of respiratory infections [p_2c34293f7e92f17e5467]. Since COVID-19 is a viral respiratory illness, it is caused by a virus, consistent with the pattern described in the passage.'}}

**demo-covid19: What is the cause of COVID-19?**

#### baseline

## Embedding + reranking RAG

Rerank the thirty candidates and select the five highest-ranked passages. The Haystack pipeline connects that selection to the prompt builder and generator.

This path uses the same prompt template and generator settings as Embedding RAG.


In [9]:
def rerank_api(question, candidates):
    # OpenRouter rerank is exposed through the OpenAI client's generic POST API.
    return client.post("/rerank", cast_to=dict[str, object], body=rerank_payload(
        tutorial.retrieval["reranker_model"], question, candidates))

@component
class RerankQuestion:
    @component.output_types(question=str)
    def run(self, question: str):
        if question != tutorial.questions()[0]["question"]:
            raise ValueError("Update demo_question in the selected input profile before changing the pipeline question.")
        tutorial.rerank(rerank_api)
        return {"question": question}

reranked_pipeline = Pipeline()
reranked_pipeline.add_component("rerank", RerankQuestion())
reranked_pipeline.add_component("generate", GenerateAnswer("reranked"))
reranked_pipeline.connect("rerank.question", "generate.question")


🚅 Components
  - rerank: RerankQuestion
  - generate: GenerateAnswer
🛤️ Connections
  - rerank.question -> generate.question (str)

Inspect the reranked five-passage context and generated answer. Expand the trace for the rendered prompt, request body, response, finish reason, token usage and measured timing.


In [10]:
if MODE != "plan":
    display(reranked_pipeline.run({"rerank": {"question": question}}))
    view.show("walkthrough", query_id=demo_query_id)


rerank: saved for CURRENT LIVE RUN 20261111T025839Z-db04c3e0ca6e


generate-reranked: saved for CURRENT LIVE RUN 20261111T025839Z-db04c3e0ca6e


{'generate': {'answer': 'The cause of COVID-19 is not explicitly stated in the provided passages. However, based on general knowledge outside the passages, COVID-19 is caused by the SARS-CoV-2 virus. The passages do not contain information about the cause of COVID-19. [p_65af6d90ff5ebade896e], [p_6482a0495822ba20641b], [p_696be041d02106a27d65], [p_d5fc92f8a485f1700c58], and [p_72663f8dc5ea39d41e04] discuss other conditions and their causes but do not mention COVID-19 or SARS-CoV-2. Therefore, the cause of COVID-19 cannot be determined from the given passages.'}}

**demo-covid19: What is the cause of COVID-19?**

#### baseline

#### reranked

## What changed?

Both answers identify SARS-CoV-2, but neither selected context mentions COVID-19 or SARS-CoV-2. This prompt does not explicitly prohibit outside knowledge, so a correct answer alone does not demonstrate evidence support or a reranking benefit.

The selected passages changed; the core answer did not. This optional example motivates the evidence-only prompts and separate retrieval, reference-overlap, and faithfulness evaluations in the experiments.


In [11]:
# Score only answers produced by this RUN_ID. Live includes RAGAS where declared.
score_run = await tutorial.score()
if score_run is not None:
    display(score_run)
view.show('scores')
tutorial.show()


.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'answers': 2,
 'run_id': '20261111T025839Z-db04c3e0ca6e',
 'table': 'scores/per_answer_scores.csv'}

query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,subgroup,diagnostic,answer_words,provider,run_id
demo-covid19,What is the cause of COVID-19?,baseline,baseline,"The cause of COVID-19 is a virus, specifically the SARS-CoV-2 virus. This is supported by the general understanding that viruses can cause infectious diseases, as evidenced by the passage stating that ""bacteria, viruses, or fungi can cause pneumonia,"" and that ""flu viruses"" and ""cold viruses"" are among the causes of respiratory infections [p_2c34293f7e92f17e5467]. Since COVID-19 is a viral respiratory illness, it is caused by a virus, consistent with the pattern described in the passage.",stop,False,0.0000,0.0312,0.0000,"The cause of COVID-19 is a virus, specifically the SARS-CoV-2 virus. This is supported by the general understanding that viruses can cause infectious diseases, as evidenced by the passage stating that ""bacteria, viruses, or fungi can cause pneumonia,"" and that ""flu viruses"" and ""cold viruses"" are among the causes of respiratory infections . Since COVID-19 is a viral respiratory illness, it is caused by a virus, consistent with the pattern described in the passage.","[""SARS-CoV-2""]",0.0000,3.1250,0.0000,0.0000,warmup,False,74.0000,Nebius,20261111T025839Z-db04c3e0ca6e
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,subgroup,diagnostic,answer_words,provider,run_id
demo-covid19,What is the cause of COVID-19?,reranked,reranked,"The cause of COVID-19 is not explicitly stated in the provided passages. However, based on general knowledge outside the passages, COVID-19 is caused by the SARS-CoV-2 virus. The passages do not contain information about the cause of COVID-19. [p_65af6d90ff5ebade896e], [p_6482a0495822ba20641b], [p_696be041d02106a27d65], [p_d5fc92f8a485f1700c58], and [p_72663f8dc5ea39d41e04] discuss other conditions and their causes but do not mention COVID-19 or SARS-CoV-2. Therefore, the cause of COVID-19 cannot be determined from the given passages.",stop,False,0.0000,0.0351,0.0000,"The cause of COVID-19 is not explicitly stated in the provided passages. However, based on general knowledge outside the passages, COVID-19 is caused by the SARS-CoV-2 virus. The passages do not contain information about the cause of COVID-19. , , , , and discuss other conditions and their causes but do not mention COVID-19 or SARS-CoV-2. Therefore, the cause of COVID-19 cannot be determined from the given passages.","[""SARS-CoV-2""]",0.0000,3.5088,0.0000,0.0000,warmup,False,69.0000,Nebius,20261111T025839Z-db04c3e0ca6e


subgroup,comparison,metric,matched_pairs,original_mean,variant_mean,mean_delta,bootstrap_low,bootstrap_high
warmup,reranked,squad_token_f1_pct,1,3.1250,3.5088,0.3838,0.3838,0.3838
warmup,reranked,answer_words,1,74.0000,69.0000,-5.0000,-5.0000,-5.0000


**LIVE run `20261111T025839Z-db04c3e0ca6e` — only this run’s answers, scores and timings.**

,squad_token_f1_pct,answer_words
arm,,
baseline,3.125000,74.0
reranked,3.508772,69.0


squad_token_f1_pct  answer_words
subgroup arm                                       
warmup   baseline            3.125000          74.0
         reranked            3.508772          69.0

,arm,answer,squad_token_f1_pct,answer_words
